# Imputation-Techniques Benchmark (Air-Quality Networks)

Benchmarks a **broad, representative set of imputation techniques** from the survey
taxonomy on the project's three air-quality networks (**Dhaka, Beijing, Delhi**) by
*reconstruction quality*: a sample of **observed** test-period cells is hidden, each
method reconstructs them, and we score RMSE/MAE/R² at the hidden cells — the same
protocol the paper uses for its imputability axis.

### What "run them all" means here (read this)
The taxonomy lists ~100 named methods, but many are survey entries with **no public
implementation** (Gray/purity KNN, RBI/IRBI, DFIC, D-ANFIS, SiMI/EMI/DMI, grey-system
fuzzy c-means, …) or are **special cases** of one engine (ARMA⊂ARIMA, Box-Jenkins⊂ARIMA,
6h/12h mean ⊂ hour-mean). This notebook runs **~35 reference implementations, one per
family**, and ships `coverage_map.csv` that lists **every** technique you named with its
disposition: `run` / `subsumed` / `skipped (reason)`. Nothing is silently dropped, and
no undefendable from-scratch reimplementations are reported as results.

### How to use
1. **Runtime → Change runtime type → GPU** (the deep family needs it).
2. Run the **Setup** cell and upload `air-transformer.zip` when prompted.
3. Run **Prepare data** (Beijing/Delhi auto-download; resumes if interrupted).
4. Leave `SMOKE_TEST = True` for a ~2-min wiring check, then set it to `False` for the
   full run (all 3 datasets × ~35 methods × MCAR+outage; ~2–4 h on a Pro GPU).
5. Run **Benchmark → Figures → Download** to get `imputation_benchmark_artifacts.zip`.

## 1. Setup — upload the repo zip, check GPU, install the extra imputers

In [ ]:
import os

# Find the repo (folder containing config.yaml). Upload air-transformer.zip if absent.
def _find_repo():
    for c in ('.', 'air-transformer'):
        if os.path.exists(os.path.join(c, 'config.yaml')):
            return c
    return None

if _find_repo() is None:
    from google.colab import files
    print('Upload air-transformer.zip (it should contain config.yaml at its root) ...')
    files.upload()
    os.system('unzip -q -o air-transformer.zip')

repo = _find_repo()
assert repo is not None, 'config.yaml not found — did the zip extract correctly?'
os.chdir(repo)
print('Working dir:', os.getcwd())

In [ ]:
import torch
print('Torch:', torch.__version__,
      '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else 'NONE — set Runtime > Change runtime type > GPU')

# Install ONLY the extra imputers. Do NOT reinstall torch/numpy/pandas (Colab ships a
# CUDA build; reinstalling breaks the GPU). A wheel that fails just demotes its method
# to "skipped" in the coverage map.
!pip install -q pypots fancyimpute scikit-fuzzy minisom
print('Extras installed.')

## 2. Prepare data — clean all three networks

Runs the project's existing cleaning scripts. Beijing (UCI) and Delhi (CPCB/Mendeley)
auto-download; every step resumes from existing files, so a disconnect is harmless.

In [ ]:
!python scripts/01_prepare_data.py  --config config.yaml
!python scripts/01b_prepare_beijing.py --config config_beijing.yaml
!python scripts/01c_prepare_delhi.py   --config config_delhi.yaml
print('All three processed parquets + scalers ready.')

## 3. Configuration

In [ ]:
SMOKE_TEST   = True            # True = 2-min wiring check (Dhaka, 5 fast methods)
HOLDOUT_RATE = 0.20            # fraction of observed test cells hidden per pattern
PATTERNS     = ('mcar', 'outage')   # cell-wise MCAR + contiguous outage blocks
SEED         = 42
INCLUDE_SLOW = True            # ARIMA, Kalman, SSA, MissForest, SVR, SOM, fuzzy c-means
INCLUDE_DEEP = True            # AE/DAE + pypots: GP-VAE, BRITS, M-RNN, GRU-D, SAITS, ...

DATASETS = [('dhaka', 'config.yaml'),
            ('beijing', 'config_beijing.yaml'),
            ('delhi', 'config_delhi.yaml')]

## 4. Run the benchmark

In [ ]:
import warnings, json, time, yaml
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, torch
from src import imputation_benchmark as ib
from src.data.dataset import build_station_arrays

device = 'cuda' if torch.cuda.is_available() else 'cpu'
cov = ib.coverage_map()
ds_list = DATASETS[:1] if SMOKE_TEST else DATASETS
only    = {'mean', 'forward_fill', 'linear_interp', 'hour_mean', 'knn'} if SMOKE_TEST else None

all_board, all_long = [], []
for name, cfgpath in ds_list:
    t0 = time.perf_counter()
    cfg = yaml.safe_load(open(cfgpath))
    proc = cfg['paths']['processed_dir']
    sc  = json.load(open(os.path.join(proc, 'scalers.json')))
    df  = pd.read_parquet(os.path.join(proc, 'all_stations.parquet'))
    stations = build_station_arrays(df, cfg, sc)
    methods, skipped = ib.build_methods(
        cfg, stations, sc, device=device, seed=SEED, only=only,
        include_slow=INCLUDE_SLOW and not SMOKE_TEST,
        include_deep=INCLUDE_DEEP and not SMOKE_TEST)
    print(f'[{name}] {len(methods)} methods, {len(skipped)} skipped (missing deps)')
    board, long = ib.run_benchmark(
        cfg, sc, methods, patterns=PATTERNS, holdout_rate=HOLDOUT_RATE, seed=SEED,
        max_stations=2 if SMOKE_TEST else None,
        max_slice_len=2000 if SMOKE_TEST else None)
    outdir = os.path.join('outputs', 'imputation_benchmark', name)
    os.makedirs(outdir, exist_ok=True)
    board.to_csv(os.path.join(outdir, 'leaderboard.csv'), index=False)
    long.to_csv(os.path.join(outdir, 'per_variable.csv'), index=False)
    pd.DataFrame(skipped).to_csv(os.path.join(outdir, 'skipped_methods.csv'), index=False)
    all_board.append(board); all_long.append(long)
    print(f'[{name}] done in {(time.perf_counter()-t0)/60:.1f} min')

board_all = pd.concat(all_board, ignore_index=True)
long_all  = pd.concat(all_long, ignore_index=True)
os.makedirs('outputs/imputation_benchmark', exist_ok=True)
board_all.to_csv('outputs/imputation_benchmark/leaderboard_all.csv', index=False)
cov.to_csv('outputs/imputation_benchmark/coverage_map.csv', index=False)
print('\nCoverage map:', cov['status'].value_counts().to_dict())

### Leaderboards (lower `overall_std_rmse` = better; `imputability` > 0 beats forward-fill)

**Sanity invariant:** `forward_fill` must show `imputability == 0.0` exactly (it *is* the
baseline). In the full run, a deep method (SAITS/BRITS) should beat forward-fill on Beijing
(known imputability +0.20) and *fail* to beat it on Dhaka (known −0.39) — the leaderboard
reproducing the paper's imputability ordering is the strongest end-to-end check.

In [ ]:
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 20)
cols = ['method','family','pm25_rmse_ugm3','overall_std_rmse','overall_r2','imputability','runtime_s']
for name in board_all['dataset'].str.lower().unique():
    sub = board_all[(board_all.dataset.str.lower()==name) & (board_all.pattern=='mcar')]
    print(f'\n===== {name.upper()} — MCAR holdout {HOLDOUT_RATE:.0%} =====')
    print(sub.sort_values('overall_std_rmse')[cols].to_string(index=False))

## 5. Figures

In [ ]:
import matplotlib.pyplot as plt
try:
    import seaborn as sns; sns.set_style('whitegrid')
except Exception:
    sns = None
figdir = 'outputs/imputation_benchmark/figures'; os.makedirs(figdir, exist_ok=True)

# (a) Per-dataset family-colored RMSE bar chart (MCAR)
for name in board_all['dataset'].str.lower().unique():
    sub = board_all[(board_all.dataset.str.lower()==name) & (board_all.pattern=='mcar')]
    sub = sub.sort_values('overall_std_rmse')
    fams = sub['family'].astype('category')
    colors = plt.cm.tab20(fams.cat.codes / max(1, fams.cat.categories.size))
    fig, ax = plt.subplots(figsize=(8, 0.32*len(sub)+1))
    ax.barh(sub['method'], sub['overall_std_rmse'], color=colors)
    ax.axvline(1.0, ls='--', c='k', lw=1, label='forward-fill')
    ax.set_xlabel('overall standardized RMSE (lower = better)')
    ax.set_title(f'{name.capitalize()} — reconstruction RMSE by method (MCAR)')
    ax.invert_yaxis(); ax.legend(); fig.tight_layout()
    fig.savefig(f'{figdir}/rmse_bar_{name}.png', dpi=150); plt.show()

In [ ]:
# (b) Cross-dataset imputability heatmap (MCAR), method x dataset
piv = (board_all[board_all.pattern=='mcar']
       .pivot_table(index='method', columns='dataset', values='imputability'))
if piv.shape[1] >= 1:
    fig, ax = plt.subplots(figsize=(1.6*piv.shape[1]+3, 0.33*len(piv)+1))
    if sns is not None:
        sns.heatmap(piv.sort_values(piv.columns[0]), annot=True, fmt='.2f',
                    center=0, cmap='RdYlGn', ax=ax, cbar_kws={'label':'imputability'})
    else:
        im = ax.imshow(piv.values, cmap='RdYlGn'); fig.colorbar(im)
    ax.set_title('Imputability vs forward-fill (>0 = beats ffill)')
    fig.tight_layout(); fig.savefig(f'{figdir}/imputability_heatmap.png', dpi=150); plt.show()

In [ ]:
# (c) MCAR vs outage difficulty per method (one dataset shown)
name = board_all['dataset'].str.lower().unique()[0]
sub = board_all[board_all.dataset.str.lower()==name]
wide = sub.pivot_table(index='method', columns='pattern', values='overall_std_rmse')
if {'mcar','outage'}.issubset(wide.columns):
    fig, ax = plt.subplots(figsize=(6,6))
    ax.scatter(wide['mcar'], wide['outage'])
    for m,r in wide.iterrows(): ax.annotate(m,(r['mcar'],r['outage']),fontsize=7)
    lim=[0,max(wide.max())*1.05]; ax.plot(lim,lim,'k--',lw=1)
    ax.set_xlabel('MCAR RMSE'); ax.set_ylabel('outage RMSE')
    ax.set_title(f'{name.capitalize()}: cell-wise vs block missingness')
    fig.tight_layout(); fig.savefig(f'{figdir}/mcar_vs_outage_{name}.png', dpi=150); plt.show()

## 6. Download artifacts

In [ ]:
import shutil
z = shutil.make_archive('imputation_benchmark_artifacts', 'zip',
                        root_dir='outputs', base_dir='imputation_benchmark')
print('Wrote', z)
print('Contains: per-dataset leaderboard.csv + per_variable.csv + skipped_methods.csv,'
      ' leaderboard_all.csv, coverage_map.csv, figures/. Unzip into the local repo root.')
try:
    from google.colab import files; files.download(z)
except Exception as e:
    print('(Not in Colab — download manually):', e)